In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
demo_with_zip = pd.read_csv(f'{RAW_DIR}/clarity/2025-05-19.done/episode_combined_2025-05-19.csv')
demo_with_zip = demo_with_zip[['patientID','zipCode5']].copy()
demo_with_zip = demo_with_zip.rename(columns={'patientID':'anon_id',
                                              'zipCode5':'zip'})
demo_with_zip

In [ ]:
df_adi = pd.read_csv(f'{DATA_DIR}/MA_2022_ADI_9 Digit Zip Code_v4_0_1.csv')
df_adi

In [ ]:
df_adi_9digit = df_adi.copy()
df_adi_9digit['ZIP_4'] = df_adi_9digit['ZIP_4'].astype(str).apply(
    lambda x: '0' + x if len(x) == 8 else x
)
df_adi_9digit = df_adi_9digit[df_adi_9digit['ZIP_4'].str.len() == 9]
df_adi_9digit['adi_score'] = pd.to_numeric(df_adi_9digit['ADI_NATRANK'], errors='coerce')
df_adi_9digit['adi_state_rank'] = pd.to_numeric(df_adi_9digit['ADI_STATERANK'], errors='coerce')
df_adi_9digit = df_adi_9digit[(df_adi_9digit['adi_score'].notna()) & (df_adi_9digit['adi_state_rank'].notna())]

df_adi_9digit['zip5'] = df_adi_9digit['ZIP_4'].str[:5]
df_avg_adi_scores = df_adi_9digit.groupby('zip5', as_index=False)[['adi_score','adi_state_rank']].mean()
df_avg_adi_scores

In [ ]:
cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
cohort

In [ ]:
# Merge cohort with zip
cohort_zip = (
    cohort.merge(demo_with_zip[['anon_id', 'zip']], on='anon_id', how='left')
         .drop_duplicates()
)
cohort_zip

In [ ]:
cohort_adi = cohort_zip.merge(df_avg_adi_scores,
                              left_on = ['zip'],
                              right_on = ['zip5'],
                              how = 'left')
cohort_adi = cohort_adi[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded',
                         'order_time_jittered_utc','adi_score','adi_state_rank']]
cohort_adi

In [ ]:
cohort_adi.to_csv(f'{WORK_DIR}/microbiology_cultures_adi_scores.csv',
                  index = False)

## Deidentification

In [ ]:
# Load the saved mappings
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = cohort_adi.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_adi_scores.csv',
                        index = False)